# Loan-level ECL and term structure

Two tables for every loan, built from a sealed run:

1. **Loan summary**: one row per loan, scenario and valuation date. It holds the loan's stage, its 12-month ECL, its lifetime ECL, PD, LGD, remaining maturity and discount rate, next to the provision the run actually booked.
2. **Term structure**: one row per loan, scenario, valuation date and default bucket. It holds the PD, LGD, EAD and discount factor of each year to maturity, plus that bucket's share of the ECL.

Nothing here is new modelling. The notebook joins tables the run already wrote and recomputes the ECL the way R8 does, `Σ m[k] × EAD[k] × LGD[k] × DF[k]`. It then checks the result against what the run booked. It reads `runs/<run_id>/` only. The one thing it writes is the export in section 7, to `notebooks/output/<run_id>/`, which git ignores.

| Section | Question |
|---|---|
| 1 | Which run, and which tolerances? |
| 2 | What does each loan bring: segment, country, rate, undrawn amount, SICR line? |
| 3 | The term structure: what enters each bucket's ECL term? |
| 4 | The loan summary: 12-month and lifetime ECL next to the booked provision |
| 5 | Controls, recomputed from the run's own tables for every scenario and valuation date |
| 6 | Caveats: end-2029 S1, S3 loans, raw and constrained values, the frozen opening |
| 7 | Export |

## Settings

`RUN_ID = "latest"` opens the newest sealed core run. A run ID opens that run. `EXPORT_YEARS` lists the valuation dates written in section 7. The default is the 2026 snapshot, where every loan has exactly one slice. From 2027 on a loan can be spread over several grade slices, and each slice is then its own row, keyed by `slice`. The controls in section 5 always cover every valuation date, whatever `EXPORT_YEARS` says. `FACILITY` is the worked example in sections 3 and 4.

In [ ]:
RUN_ID = "latest"  # "latest" or a run ID under runs/
EXPORT_YEARS = [2026]  # valuation dates to export: any of 2026-2029
EXPORT = True  # False skips writing the files in section 7
FACILITY = "F08156"  # worked example; section 2 shows what it is

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import nbtools as nb

nb.setup()
pd.set_option("display.max_columns", 60)

## 1 · The run

The run folder holds everything used here: the input copy in `data/` and the results in `results/`. Every tolerance below comes from the run's own parameter set, so no limit is typed into this notebook.

In [ ]:
run = nb.open_run(RUN_ID)
ps = run.data("parameter_set")
grades = run.grades()
rank = {g: i + 1 for i, g in enumerate(grades)}
SNAPSHOT = int(ps["horizon"]["snapshot_year"])
LAST = int(max(ps["horizon"]["projection_years"]))  # end of the horizon: S1 uses the blend here
TOL_P = float(ps["tolerances"]["probability_identity"]["abs"])  # probabilities
TOL_RATE = float(ps["tolerances"]["rate"]["abs"])  # discount factors
TOL_EUR = float(ps["tolerances"]["amount"]["abs"])  # one slice's amount
AGG = ps["tolerances"]["aggregate"]  # sums: max(abs, rel x amount)

display(run.overview().to_frame())
pd.Series({"probability": f"{TOL_P:g}", "rate": f"{TOL_RATE:g}", "amount": f"{TOL_EUR:g} EUR",
           "aggregate": f"max({AGG['abs']:g} EUR, {AGG['rel']:g} x amount)"},
          name="tolerance").to_frame()

The run's tables, read once. `slices` holds each loan's grade slices per year, `parameters` the LGD, prepayment, EAD and discount factor per facility and default bucket, `pd_curves` the PD curve of every segment, country and grade, `cf_schedule` the contractual schedule, `provisions` what R8 booked and `drivers` the Z-scores with the terminal weights. PD, LGD and EAD in one ECL term must share their event basis (ECL-08), so the cell checks that each scenario and valuation date has one basis only, the same in `parameters` and `pd_curves`.

In [ ]:
slices = run.table("slices")
params = run.table("parameters")
curves = run.table("pd_curves")
schedule = run.table("cf_schedule")
provisions = run.table("provisions")
ecl_buckets = run.table("ecl_buckets")
lgd_year = run.table("lgd_of_year")
drivers = run.table("drivers")

for name, table in (("parameters", params), ("pd_curves", curves)):
    bases = table.groupby(["scenario", "valuation_year"])["basis"].nunique()
    assert (bases == 1).all(), f"{name}: more than one event basis in a scenario and valuation date"
assert set(params["basis"]) == set(curves["basis"]), "parameters and pd_curves differ in basis"
YEARS = sorted(int(y) for y in slices["year"].unique())
assert set(EXPORT_YEARS) <= set(YEARS), f"EXPORT_YEARS must be among the run's years {YEARS}"

pd.DataFrame({name: [f"{len(df):,}"] for name, df in [
    ("slices", slices), ("parameters", params), ("pd_curves", curves), ("cf_schedule", schedule),
    ("provisions", provisions), ("ecl_buckets", ecl_buckets), ("lgd_of_year", lgd_year)]},
    index=["rows"])

## 2 · What each loan brings

From the input copy: segment and portfolio, the obligor's country (the PD curves are per segment and country), the interest rate, the drawn amount (GCA) and the undrawn commitment. The interest rate is used as the effective interest rate and held fixed for the loan's whole life, floating-rate loans included (D12). No results table carries it, so it comes from `facilities`.

**SICR line.** No results table carries it either, so it is rebuilt here by hand from the TTC matrix, as MIG-07 states it. A grade is S2 for a loan when its lifetime TTC PD is at least k times the lifetime TTC PD at the loan's origination grade. Both run over the loan's residual maturity at the snapshot (assumption A6), with the constant-hazard rule for a part-year (PD-06). `sicr_grade` is the best grade that is S2 for the loan. It is empty when no performing grade reaches the line, and such a loan can never become S2. Section 5 checks the rebuilt line against every stage in `slices`.

In [ ]:
book = run.book()  # facilities with the obligor's country and sector
undrawn = run.data("commitments").groupby("facility_id")["nominal_undrawn"].sum()
facts = book[["facility_id", "portfolio", "segment", "country", "product", "ifrs_stage", "gca",
              "interest_rate", "rate_type", "residual_maturity", "current_grade",
              "origination_grade", "ttc_lgd"]].rename(columns={"gca": "facility_gca"})
facts["facility_undrawn"] = facts["facility_id"].map(undrawn).fillna(0.0)

ttc = {}
for segment, cells in run.data("ttc_pd_matrix").groupby("segment"):
    m = np.zeros((len(grades) + 1, len(grades) + 1))
    m[cells["from_rank"] - 1, cells["to_rank"] - 1] = cells["probability"]
    m[-1, -1] = 1.0  # D is absorbing
    ttc[segment] = m


def lifetime_ttc(segment: str, horizon: float) -> np.ndarray:
    # Lifetime TTC PD of every performing grade over the horizon; a stub uses 1 - (1 - PD)^d.
    whole = int(np.floor(horizon + 1e-9))
    stub = horizon - whole
    power = np.linalg.matrix_power(ttc[segment], whole)
    life = power[:-1, -1].copy()
    if stub > 1e-9:
        life += power[:-1, :-1] @ (1.0 - (1.0 - ttc[segment][:-1, -1]) ** stub)
    return life


def sicr_grade(segment: str, horizon: float, origination: str):
    life = lifetime_ttc(segment, horizon)
    s2 = life >= float(ps["segments"][segment]["sicr_k"]) * life[rank[origination] - 1]
    return grades[int(np.argmax(s2))] if s2.any() else None


performing = facts["ifrs_stage"] < 3
facts["sicr_grade"] = None
facts.loc[performing, "sicr_grade"] = [
    sicr_grade(s, h, g) for s, h, g in facts.loc[performing, ["segment", "residual_maturity",
                                                              "origination_grade"]].to_numpy()]
never_s2 = performing & facts["sicr_grade"].isna()
print(f"{len(facts):,} loans; {int(never_s2.sum()):,} performing loans have no S2 grade "
      f"(EUR {facts.loc[never_s2, 'facility_gca'].sum() / nb.MILLION:,.1f}m GCA)")
assert FACILITY in set(facts["facility_id"]), f"{FACILITY} is not in this run"
facts[facts["facility_id"] == FACILITY].T

## 3 · The term structure

One row per performing slice and default bucket, for one scenario and valuation date. The rows come from three joins:

- `slices` → the slice's stage, grade and amounts.
- `parameters`, on facility, scenario, valuation date and bucket → LGD (point-in-time, from the Z of the bucket year, with no recovery discounting, LGD-12), the prepayment rate and the share not yet prepaid, the contractual balance, drawn and undrawn EAD (conditional on default, prepayment already applied, EAD-11), and the discount factor `DF = (1 + r)^-t` from the default date back to the valuation date.
- `pd_curves`, on segment, country, scenario, valuation date, grade and bucket → q (conditional PD), m (marginal PD), S (survival) and F (cumulative PD). The curve is stored per segment, country and grade, not per loan, and it holds whole years only. A final bucket of d years (`fraction < 1`) takes d × its whole-year marginal PD, pro rata by the maturity, as R8 does (A7). Its F, S and q follow from that: F = F[n] + m, S = 1 − F and q = m ÷ S[n].

Parameters are per facility, but a slice holds only part of the facility. Its terms are therefore scaled by the slice's share: GCA for the drawn part and the undrawn amount for the off-balance part, as R8 does. At the 2026 snapshot every share is 1.

`ecl_on = m × EAD_drawn × LGD × DF × share_on` goes to the on-balance allowance, and `ecl_off` (undrawn EAD) to the off-balance provision. `ecl_share` is the bucket's share of the slice's lifetime ECL.

In [ ]:
CURVE_KEY = ["segment", "country", "scenario", "valuation_year", "grade", "default_bucket"]
PARAM_COLUMNS = ["facility_id", "default_bucket", "fraction", "bucket_year", "t", "z", "lgd",
                 "prepayment_rate", "not_prepaid_share", "balance", "ccf", "ead_drawn",
                 "ead_undrawn", "ead_total", "discount_factor"]


def term_structure(scenario: str, year: int) -> pd.DataFrame:
    part = slices[(slices["scenario"] == scenario) & (slices["year"] == year)
                  & (slices["stage"] < 3)]
    ts = part[["scenario", "year", "facility_id", "slice", "stage", "grade", "gca", "undrawn"]]
    ts = ts.rename(columns={"year": "valuation_year", "gca": "slice_gca",
                            "undrawn": "slice_undrawn"})
    ts = ts.merge(facts[["facility_id", "segment", "country", "interest_rate", "facility_gca",
                         "facility_undrawn"]], on="facility_id", how="left", validate="many_to_one")
    par = params[(params["scenario"] == scenario) & (params["valuation_year"] == year)]
    ts = ts.merge(par[PARAM_COLUMNS], on="facility_id", how="left", validate="many_to_many")
    cf = schedule[schedule["valuation_year"] == year]
    cf = cf[["facility_id", "period", "period_end", "remaining_maturity"]]
    ts = ts.merge(cf.rename(columns={"period": "default_bucket"}),
                  on=["facility_id", "default_bucket"], how="left", validate="many_to_one")
    cv = curves[(curves["scenario"] == scenario) & (curves["valuation_year"] == year)]
    cv = cv.rename(columns={"cohort": "grade"})[[*CURVE_KEY, "q", "m", "s", "f"]]
    ts = ts.merge(cv, on=CURVE_KEY, how="left", validate="many_to_one")
    if ts[["m", "ead_drawn", "period_end"]].isna().any().any():
        raise ValueError(f"{scenario} {year}: a bucket has no PD curve, parameter or schedule row")
    # A final bucket of d years takes d of its whole-year marginal PD (A7); F, S and q follow.
    part = ts["fraction"] < 1.0
    before = ts.loc[part, "f"] - ts.loc[part, "m"]  # F at the end of the last whole year
    ts.loc[part, "m"] = ts.loc[part, "m"] * ts.loc[part, "fraction"]
    ts.loc[part, "f"] = before + ts.loc[part, "m"]
    ts.loc[part, "s"] = 1.0 - ts.loc[part, "f"]
    ts.loc[part, "q"] = ts.loc[part, "m"] / (1.0 - before)
    ts["share_on"] = np.divide(ts["slice_gca"], ts["facility_gca"], out=np.zeros(len(ts)),
                               where=ts["facility_gca"] > 0)
    ts["share_off"] = np.divide(ts["slice_undrawn"], ts["facility_undrawn"],
                                out=np.zeros(len(ts)), where=ts["facility_undrawn"] > 0)
    ts["ecl_on"] = ts["m"] * ts["ead_drawn"] * ts["lgd"] * ts["discount_factor"] * ts["share_on"]
    ts["ecl_off"] = (ts["m"] * ts["ead_undrawn"] * ts["lgd"] * ts["discount_factor"]
                     * ts["share_off"])
    life = ts.groupby(["facility_id", "slice"])[["ecl_on", "ecl_off"]].transform("sum").sum(axis=1)
    ts["ecl_share"] = np.divide(ts["ecl_on"] + ts["ecl_off"], life, out=np.full(len(ts), np.nan),
                                where=life > 0)
    ts["in_12m"] = ts["default_bucket"] == 1
    ts = ts.drop(columns=["facility_gca", "facility_undrawn"])
    return ts.sort_values(["facility_id", "slice", "default_bucket"], kind="mergesort",
                          ignore_index=True)


example = term_structure("adverse", SNAPSHOT)
example = example[example["facility_id"] == FACILITY]
example[["default_bucket", "bucket_year", "period_end", "fraction", "t", "z", "q", "m", "s", "f",
         "lgd", "not_prepaid_share", "ead_drawn", "ead_undrawn", "discount_factor", "ecl_on",
         "ecl_off", "ecl_share"]]

The worked example under both scenarios at the snapshot: each bucket's ECL contribution. The bars to the right of bucket 1 are what lifetime adds to the 12-month ECL. The last bucket is a part-year, so its PD and its contribution are smaller.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
width = 0.4
for i, scenario in enumerate(run.scenarios):
    one = term_structure(scenario, SNAPSHOT)
    one = one[one["facility_id"] == FACILITY]
    ax.bar(one["bucket_year"] + (i - 0.5) * width, (one["ecl_on"] + one["ecl_off"]) / 1e3,
           width=width, color=nb.SCENARIO_COLOURS[scenario], label=scenario)
ax.set_xlabel("default bucket year")
ax.set_ylabel("ECL term, EUR thousand")
ax.set_title(f"{FACILITY}: ECL by default bucket, valued at end-{SNAPSHOT}")
ax.legend()
plt.show()

## 4 · The loan summary

One row per slice (one per loan at the snapshot), for one scenario and valuation date.

- **12-month ECL**: bucket 1 only, on and off balance. When bucket 1 is a part-year (maturity under a year), it covers only that part-year.
- **Lifetime ECL**: the sum of every bucket to maturity.
- Both are given for every performing loan, whatever its stage. `recomputed_on` / `recomputed_off` pick the one the stage calls for: 12-month for S1, lifetime for S2.
- `pd_12m` is m[1], `pd_lifetime` is F at maturity, and `lgd_1`, `df_1`, `ead_drawn_1` and `ead_undrawn_1` belong to bucket 1.
- **Booked**: `method`, `raw_on` / `raw_off` (before floors) and `allowance_on` / `provision_off` (after floors) from `provisions`. `diff_on` / `diff_off` are booked raw minus recomputed.
- **End of the horizon (end-2029), S1**: R8 books the interpolated loss rate `a × PD × LGD + (1 − a) × PD_base × LGD_base` (ECL-10, AT-06), with each scenario's one-year PD of the last scenario year (bucket 1 of the previous year-end's curve), its LGD of that year and the terminal weight `a` of the next year from `drivers`. A part-year first bucket takes d × each PD first (A27). `terminal_blend_on` / `terminal_blend_off` recompute that value here. `terminal_s1` marks these rows, where `diff_on` / `diff_off` therefore measure the blend against the curve.
- At the snapshot, `provisions` holds the engine's own valuation of the opening, a diagnostic (A16). The charges start from the frozen O4 allowance instead, shown as `opening_o4_on` / `opening_o4_off`.
- **S3 slices** have stage 3, PD = 1 and the allowance as booked. They have no 12-month or lifetime split and no term structure. Their LGD is the TTC LGD at the snapshot (O4) and the LGD of the year after it (ECL-11).

In [ ]:
SUMMARY_COLUMNS = [
    "scenario", "valuation_year", "facility_id", "slice", "portfolio", "segment", "country",
    "product", "stage", "grade", "origination_grade", "sicr_grade", "slice_gca", "slice_undrawn",
    "interest_rate", "rate_type", "remaining_maturity", "buckets", "fraction_1", "last_fraction",
    "pd_12m", "pd_lifetime", "lgd_1", "df_1", "ead_drawn_1", "ead_undrawn_1", "ecl12_on",
    "ecl12_off", "ecl_life_on", "ecl_life_off", "recomputed_on", "recomputed_off", "method",
    "raw_on", "raw_off", "allowance_on", "provision_off", "floor_binding", "diff_on", "diff_off",
    "terminal_s1", "terminal_blend_on", "terminal_blend_off", "opening_o4_on", "opening_o4_off",
    "entry_year", "origin_stage"]


def performing_summary(ts: pd.DataFrame) -> pd.DataFrame:
    by_slice = ts.groupby(["facility_id", "slice"], sort=False)
    first = ts[ts["in_12m"]].set_index(["facility_id", "slice"])
    last = ts.loc[by_slice["default_bucket"].idxmax()].set_index(["facility_id", "slice"])
    out = first[["scenario", "valuation_year", "stage", "grade", "slice_gca", "slice_undrawn",
                 "interest_rate", "remaining_maturity"]].copy()
    out["buckets"] = last["default_bucket"]
    out["fraction_1"] = first["fraction"]
    out["last_fraction"] = last["fraction"]
    out["pd_12m"] = first["m"]
    out["pd_lifetime"] = last["f"]
    out["lgd_1"] = first["lgd"]
    out["df_1"] = first["discount_factor"]
    out["ead_drawn_1"] = first["ead_drawn"]
    out["ead_undrawn_1"] = first["ead_undrawn"]
    out["ecl12_on"] = first["ecl_on"]
    out["ecl12_off"] = first["ecl_off"]
    lifetime = by_slice[["ecl_on", "ecl_off"]].sum()
    out["ecl_life_on"] = lifetime["ecl_on"]
    out["ecl_life_off"] = lifetime["ecl_off"]
    s1 = out["stage"] == 1
    out["recomputed_on"] = np.where(s1, out["ecl12_on"], out["ecl_life_on"])
    out["recomputed_off"] = np.where(s1, out["ecl12_off"], out["ecl_life_off"])
    return out.reset_index()


def s3_summary(scenario: str, year: int) -> pd.DataFrame:
    part = slices[(slices["scenario"] == scenario) & (slices["year"] == year)
                  & (slices["stage"] == 3)]
    out = part[["scenario", "year", "facility_id", "slice", "stage", "grade", "gca", "undrawn",
                "entry_year", "origin_stage"]]
    out = out.rename(columns={"year": "valuation_year", "gca": "slice_gca",
                              "undrawn": "slice_undrawn"})
    if year == SNAPSHOT:
        lgd = facts.set_index("facility_id")["ttc_lgd"]  # O4 values the opening S3 at TTC LGD
    else:
        lgd = lgd_year[(lgd_year["scenario"] == scenario) & (lgd_year["year"] == year)]
        lgd = lgd.set_index("facility_id")["lgd"]
    return out.assign(lgd_1=out["facility_id"].map(lgd), pd_12m=1.0, pd_lifetime=1.0)


def loss_rate(scenario: str, first: pd.DataFrame) -> np.ndarray:
    # One-year PD of the last scenario year x LGD of that year, per slice (part-year pro rata).
    one_year = curves[(curves["scenario"] == scenario) & (curves["valuation_year"] == LAST - 1)
                      & (curves["default_bucket"] == 1)]
    one_year = one_year.rename(columns={"cohort": "grade"})[["segment", "country", "grade", "m"]]
    x = first[["segment", "country", "grade"]].merge(one_year, how="left", validate="many_to_one")
    lgd = lgd_year[(lgd_year["scenario"] == scenario) & (lgd_year["year"] == LAST)]
    lgd = first["facility_id"].map(lgd.set_index("facility_id")["lgd"]).to_numpy()
    pd_ = x["m"].to_numpy() * first["fraction"].to_numpy()  # pro rata in a part-year (A27)
    if np.isnan(pd_).any() or np.isnan(lgd).any():
        raise ValueError(f"{scenario} {LAST}: PD or LGD missing for the terminal loss rate")
    return pd_ * lgd


def terminal_blend(ts: pd.DataFrame, scenario: str) -> pd.DataFrame:
    # End-of-horizon S1 as R8 books it (ECL-10, AT-06): blend of the two scenarios' loss rates.
    first = ts[ts["in_12m"] & (ts["stage"] == 1)].reset_index(drop=True)
    weights = drivers[(drivers["scenario"] == scenario) & (drivers["year"] == LAST + 1)]
    weights = weights[["segment", "country", "adverse_weight"]]
    a = first[["segment", "country"]].merge(weights, how="left", validate="many_to_one")
    a = a["adverse_weight"].to_numpy()
    rate = a * loss_rate(scenario, first) + (1.0 - a) * loss_rate("baseline", first)
    return first[["facility_id", "slice"]].assign(
        terminal_blend_on=rate * first["ead_drawn"] * first["discount_factor"] * first["share_on"],
        terminal_blend_off=rate * first["ead_undrawn"] * first["discount_factor"]
        * first["share_off"])


def loan_tables(scenario: str, year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Term structure and loan summary of one scenario and valuation date.
    ts = term_structure(scenario, year)
    out = pd.concat([performing_summary(ts), s3_summary(scenario, year)], ignore_index=True)
    out = out.merge(facts[["facility_id", "portfolio", "segment", "country", "product",
                           "rate_type", "origination_grade", "sicr_grade"]],
                    on="facility_id", how="left", validate="many_to_one")
    out["interest_rate"] = out["facility_id"].map(facts.set_index("facility_id")["interest_rate"])
    booked = provisions[(provisions["scenario"] == scenario) & (provisions["year"] == year)]
    booked = booked[["facility_id", "slice", "method", "raw_on", "raw_off", "allowance_on",
                     "provision_off", "floor_binding"]]
    out = out.merge(booked, on=["facility_id", "slice"], how="left", validate="one_to_one")
    if out["method"].isna().any():
        raise ValueError(f"{scenario} {year}: a slice has no booked provision")
    opening = slices[(slices["scenario"] == scenario) & (slices["year"] == SNAPSHOT)]
    opening = opening.set_index(["facility_id", "slice"])
    key = pd.MultiIndex.from_frame(out[["facility_id", "slice"]])
    at_snapshot = year == SNAPSHOT
    out["opening_o4_on"] = opening["allowance_on"].reindex(key).to_numpy() if at_snapshot else np.nan
    out["opening_o4_off"] = opening["provision_off"].reindex(key).to_numpy() if at_snapshot else np.nan
    out["diff_on"] = out["raw_on"] - out["recomputed_on"]
    out["diff_off"] = out["raw_off"] - out["recomputed_off"]
    out["terminal_s1"] = out["method"] == "s1_terminal_interpolated"
    if year == LAST:
        out = out.merge(terminal_blend(ts, scenario), on=["facility_id", "slice"], how="left",
                        validate="one_to_one")
    else:
        out = out.assign(terminal_blend_on=np.nan, terminal_blend_off=np.nan)
    out = out[SUMMARY_COLUMNS].sort_values(["facility_id", "slice"], kind="mergesort")
    return ts, out.reset_index(drop=True)


_, example = loan_tables("adverse", SNAPSHOT)
example[example["facility_id"] == FACILITY].T

## 5 · Controls

Each control recomputes an identity from the run's own tables, for both scenarios and every valuation date from 2026 to 2029. No expected value is typed here.

| Control | What must hold |
|---|---|
| S1 recomputed = raw | the recomputed 12-month ECL of every S1 slice equals `raw_on` / `raw_off`, 2026–2028 |
| S1 end-2029 blend = raw | the loss-rate blend recomputed in section 4 equals `raw_on` / `raw_off` of every end-2029 S1 slice |
| S1 end-2029 curve = blend, baseline | with `a = 1` the baseline blend is its own 2029 loss rate, and the baseline Z stays flat after 2029, so the curve gives the same value. In the adverse the two differ by design; that row is listed as an expected difference and section 6 sizes it |
| S2 recomputed = raw | the recomputed lifetime ECL of every S2 slice equals `raw_on` / `raw_off`, every year |
| Bucket sums = `ecl_buckets` | the bucket terms summed by segment, stage and bucket equal R8's `ecl_buckets` (S1 counts bucket 1 only, as R8 does; at end-2029 S2 only, as S1 is the blend there) |
| Curves hold whole years | every PD curve runs from bucket 1 without a gap and has no `fraction` column (A7) |
| F + S = 1, F non-decreasing | on every PD curve, and on every slice's term structure, its part-year bucket included |
| DF = (1 + r)^-t | for every bucket row, with r the loan's interest rate |
| Stage = SICR rule | every performing slice's stage follows from its grade and the SICR line of section 2 (MIG-07) |

The `measure` column holds what the `measure is` column names: the largest absolute difference for the comparisons, the smallest step of F for the monotonicity checks, and the count of mismatches for the stage rule.

In [ ]:
def check(name, scenario, year, rows, measure, what, ok):
    return {"control": name, "scenario": scenario, "valuation": year, "rows": rows,
            "measure": measure, "measure is": what, "pass": ok}


def worst_gap(frame: pd.DataFrame, on: str, off: str) -> float:
    return max((frame["raw_on"] - frame[on]).abs().max(), (frame["raw_off"] - frame[off]).abs().max())


def within_aggregate(actual: pd.Series, expected: pd.Series) -> pd.Series:
    return (actual - expected).abs() <= np.maximum(AGG["abs"], AGG["rel"] * expected.abs())


EUR_GAP = "max abs diff, EUR"
controls, exports = [], {}
for scenario in run.scenarios:
    for year in YEARS:
        ts, summary = loan_tables(scenario, year)
        perf = summary[summary["stage"] < 3]
        for stage in (1, 2):
            part = perf[(perf["stage"] == stage) & ~perf["terminal_s1"]]
            if len(part):
                worst = worst_gap(part, "recomputed_on", "recomputed_off")
                controls.append(check(f"S{stage} recomputed = raw", scenario, year, len(part),
                                      worst, EUR_GAP, worst <= TOL_EUR))
        terminal = perf[perf["terminal_s1"]]
        if len(terminal):
            worst = worst_gap(terminal, "terminal_blend_on", "terminal_blend_off")
            controls.append(check(f"S1 end-{LAST} blend = raw", scenario, year, len(terminal),
                                  worst, EUR_GAP, worst <= TOL_EUR))
            worst = worst_gap(terminal, "recomputed_on", "recomputed_off")
            flat = bool((drivers.loc[drivers["scenario"] == scenario, "adverse_weight"] == 1).all())
            name = f"S1 end-{LAST} curve = blend" if flat else f"S1 end-{LAST} curve vs blend"
            controls.append(check(name + ("" if flat else " (expected to differ)"), scenario,
                                  year, len(terminal), worst, EUR_GAP,
                                  worst <= TOL_EUR if flat else None))
        mine = ts[(ts["stage"] == 2) | ts["in_12m"]]
        mine = mine.groupby(["segment", "stage", "default_bucket"])[["ecl_on", "ecl_off"]].sum()
        engine = ecl_buckets[(ecl_buckets["scenario"] == scenario)
                             & (ecl_buckets["valuation_year"] == year)]
        engine = engine.set_index(["segment", "stage", "default_bucket"])[["ecl_on", "ecl_off"]]
        both = mine.join(engine, how="outer", rsuffix="_engine").fillna(0.0)
        if len(terminal):
            both = both[both.index.get_level_values("stage") == 2]
        ok = (within_aggregate(both["ecl_on"], both["ecl_on_engine"])
              & within_aggregate(both["ecl_off"], both["ecl_off_engine"]))
        worst = max((both["ecl_on"] - both["ecl_on_engine"]).abs().max(),
                    (both["ecl_off"] - both["ecl_off_engine"]).abs().max())
        controls.append(check("bucket sums = ecl_buckets", scenario, year, len(both), worst,
                              EUR_GAP, bool(ok.all())))
        df_gap = (ts["discount_factor"] - (1.0 + ts["interest_rate"]) ** (-ts["t"])).abs().max()
        controls.append(check("DF = (1 + r)^-t", scenario, year, len(ts), df_gap,
                              "max abs diff", df_gap <= TOL_RATE))
        fs_ts = (ts["f"] + ts["s"] - 1.0).abs().max()
        step_ts = ts.groupby(["facility_id", "slice"])["f"].diff().min()
        controls.append(check("term structure: F + S = 1, F non-decreasing", scenario, year,
                              len(ts), step_ts, "smallest step of F",
                              fs_ts <= TOL_P and step_ts >= -TOL_P))
        line = perf["sicr_grade"].map(rank).fillna(len(grades) + 1)
        rule = np.where(perf["grade"].map(rank) >= line, 2, 1)
        wrong = int((rule != perf["stage"]).sum())
        controls.append(check("stage = SICR rule (MIG-07)", scenario, year, len(perf), wrong,
                              "slices that disagree", wrong == 0))
        if year in EXPORT_YEARS:
            exports[(scenario, year)] = (ts, summary)

curve_id = ["segment", "country", "scenario", "valuation_year", "cohort"]
c = curves.sort_values([*curve_id, "default_bucket"], kind="mergesort")
span = c.groupby(curve_id)["default_bucket"].agg(["min", "max", "count"])
gaps = int(((span["min"] != 1) | (span["count"] != span["max"])).sum())
controls.append(check("curves hold whole years (A7)", "all", "all", len(c), gaps,
                      "curves with a gap", gaps == 0 and "fraction" not in c.columns))
fs_gap = (c["f"] + c["s"] - 1.0).abs().max()
step = c.groupby(curve_id)["f"].diff().min()
controls.append(check("F + S = 1", "all", "all", len(c), fs_gap, "max abs diff", fs_gap <= TOL_P))
controls.append(check("F non-decreasing", "all", "all", len(c), step, "smallest step of F",
                      step >= -TOL_P))

controls = pd.DataFrame(controls)
failed = controls[controls["pass"].eq(False)]
print(f"{len(controls)} controls: {int(controls['pass'].eq(True).sum())} pass, {len(failed)} fail, "
      f"{int(controls['pass'].isna().sum())} expected differences (end-2029 S1)")
controls

In [ ]:
assert failed.empty, f"{len(failed)} controls failed:\n{failed}"

## 6 · Caveats

### End-2029 S1: booked blend against the curve

At end-2029 the engine books S1 from a blend of 2029 loss rates, `a × PD_adv × LGD_adv + (1 − a) × PD_base × LGD_base` with `a = 5/6` (ECL-10, AT-06), not from `m[1] × LGD[1]` of the curve. This is the formula of Boxes 4 and 5 of the EBA's 2027 draft methodological note (11 June 2026, p. 53). The recomputed 12-month ECL from the curve at 2029 therefore differs from the booked provision. Both are kept side by side here, and section 5 checks that the booked value equals the blend. In the baseline `a = 1`, so the two agree.

In [ ]:
rows = []
for scenario in run.scenarios:
    _, summary = loan_tables(scenario, LAST)
    s1 = summary[summary["terminal_s1"]].copy()
    s1["booked"] = s1["raw_on"] + s1["raw_off"]
    s1["curve"] = s1["recomputed_on"] + s1["recomputed_off"]
    part = s1.groupby("segment")[["booked", "curve"]].sum()
    part.loc["Total"] = part.sum()
    part["difference"] = part["booked"] - part["curve"]
    part["difference %"] = 100 * part["difference"] / part["curve"]
    rows.append(part.assign(scenario=scenario).reset_index())
terminal_gap = pd.concat(rows).set_index(["scenario", "segment"])
nb.millions(terminal_gap, ["booked", "curve", "difference"], digits=2).format(
    "{:+.2f}%", subset=["difference %"])

### S3 loans

S3 has no term structure. The allowance is `GCA × LGD of the year`, floored at the prior allowance per exposure (ECL-11). A new S3 cohort is booked once in its entry year at `flow × LGD of the default year`. At the snapshot the S3 allowance is O4's, `GCA × TTC LGD`.

### Raw and constrained values

A floor can lift the booked value above the raw one. Only S3 slices carry a floor in v1. The table counts the slices whose floor binds and the amount it masks, per scenario and valuation date.

In [ ]:
masked = provisions.assign(masked=provisions["masked_on"] + provisions["masked_off"])
masked = masked.groupby(["scenario", "year", "stage"]).agg(
    slices=("slice", "size"), floor_binding=("floor_binding", "sum"),
    masked_eur=("masked", "sum"))
nb.millions(masked, ["masked_eur"], digits=2)

### The frozen opening against the engine's own view

The charges start from the frozen O4 allowance. O4 values the opening book with the TTC matrix itself, while the engine's own end-2026 valuation uses the point-in-time matrices of the scenario (A8, A16, C2). The gap between the two lands in the first projected year's charge. Per loan it sits in the snapshot rows as `opening_o4_*` against `raw_*`.

In [ ]:
rows = []
for scenario in run.scenarios:
    _, summary = loan_tables(scenario, SNAPSHOT)
    summary = summary.assign(o4=summary["opening_o4_on"] + summary["opening_o4_off"],
                             engine=summary["raw_on"] + summary["raw_off"])
    part = summary.groupby("stage")[["o4", "engine"]].sum()
    part.loc["Total"] = part.sum()
    rows.append(part.assign(gap=part["engine"] - part["o4"], scenario=scenario).reset_index())
opening_gap = pd.concat(rows).set_index(["scenario", "stage"])
nb.millions(opening_gap)

## 7 · Export

Both tables for the valuation dates in `EXPORT_YEARS`, both scenarios, as Parquet (unrounded float64, every column), plus the loan summary as CSV for a spreadsheet. They go to `notebooks/output/<run_id>/`, which git ignores. Nothing is written under `runs/` or `inputs/`.

In [ ]:
if EXPORT:
    out_dir = Path(nb.ROOT) / "notebooks" / "output" / run.run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    years = "-".join(str(y) for y in EXPORT_YEARS)
    term = pd.concat([exports[k][0] for k in sorted(exports)], ignore_index=True)
    summary = pd.concat([exports[k][1] for k in sorted(exports)], ignore_index=True)
    files = {
        out_dir / f"loan_term_structure_{years}.parquet": term,
        out_dir / f"loan_summary_{years}.parquet": summary,
    }
    for path, table in files.items():
        table.to_parquet(path, index=False)
    summary.to_csv(out_dir / f"loan_summary_{years}.csv", index=False)
    display(pd.DataFrame([{"file": p.relative_to(nb.ROOT).as_posix(),
                           "rows": len(pd.read_parquet(p)) if p.suffix == ".parquet" else len(summary),
                           "MB": round(p.stat().st_size / 1e6, 1)}
                          for p in sorted(out_dir.glob(f"loan_*_{years}.*"))]))